# KeelNet Stage 6: Adaptive Constraint Balancing Template

Use this notebook as the Stage 6 working file for the official team workflow:

1. edit locally in VS Code
2. open this notebook in browser Google Colab
3. rerun the setup cell after pushing code changes
4. save artifacts to Google Drive or your local runtime project folder

This notebook keeps the Stage 6 notes, implementation hints, run commands, and shareable outputs in one place.


## Stage Notes

### Goal

Learn or control the trade-off among answer quality, support, abstention, and confidence more intelligently than fixed weights alone.

### Proof Status

This stage is not proof that hallucination is solved. It is the full-pipeline validation stage.

### Scope

- input: signals from the earlier stages
- output: adaptive balancing or decision control

### Main Change

- replace purely fixed balancing with an adaptive mechanism

### Main Metrics

- trade-off curve quality
- comparison against fixed-weight baselines
- robustness across operating points

### What This Stage Validates

- the complete KeelNet pipeline can manage the main constraints better than fixed balancing
- adaptive control adds real value beyond manual tuning

### Handoff Condition

This stage is only justified if the adaptive method clearly beats simpler fixed-weight or threshold baselines, and the gain is not just threshold gaming or over-abstention.


<h2 style="color: #1d4ed8;">1. Setup And Sync</h2>

Run this cell in either hosted Google Colab or Google Colab connected to a local Jupyter runtime.

What it does:

- hosted Colab: mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the stage branch, and installs the project
- local runtime: reuses your local repo, uses a local project folder for artifacts, reads `HF_TOKEN` from the environment if available, and installs the project into the current kernel environment

Path reminder:

- hosted Colab defaults: repo `/content/KeelNet`, project folder `/content/drive/MyDrive/KeelNet`
- local runtime defaults: repo `/content/KeelNet` if present, otherwise your current local checkout; project folder `/content/KeelNet-local`
- optional overrides: `KEELNET_REPO_DIR`, `KEELNET_PROJECT_DIR`, and `KEELNET_DRIVE_SYNC_DIR`


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys


GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
DEFAULT_GIT_BRANCH = 'main'
GIT_BRANCH = os.environ.get("KEELNET_GIT_BRANCH", DEFAULT_GIT_BRANCH)
HOSTED_COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")
DEFAULT_LOCAL_PROJECT_DIR = Path("/content/KeelNet-local")
DEFAULT_LOCAL_REPO_DIR = Path("/content/KeelNet")


def detect_runtime_mode() -> str:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return "local-runtime"
    return "hosted-colab"


RUNTIME_MODE = detect_runtime_mode()
IS_HOSTED_COLAB = RUNTIME_MODE == "hosted-colab"
PROJECT_STORAGE_LABEL = "Drive project dir" if IS_HOSTED_COLAB else "Local project dir"


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def configure_project_storage() -> Path:
    if IS_HOSTED_COLAB:
        from google.colab import drive

        project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(HOSTED_COLAB_PROJECT_DIR)))
        drive.mount("/content/drive", force_remount=False)
        if not str(project_dir).startswith("/content/drive/"):
            raise ValueError(
                f"KEELNET_PROJECT_DIR must point inside /content/drive in hosted Colab, got: {project_dir}"
            )
        project_dir.mkdir(parents=True, exist_ok=True)
        print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
        return project_dir

    project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(DEFAULT_LOCAL_PROJECT_DIR))).expanduser().resolve()
    project_dir.mkdir(parents=True, exist_ok=True)
    print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
    return project_dir


def configure_drive_project_dir(project_storage_dir: Path) -> Path | None:
    if IS_HOSTED_COLAB:
        print(f"Drive project dir: {project_storage_dir}")
        return project_storage_dir.resolve()

    env_drive_dir = os.environ.get("KEELNET_DRIVE_SYNC_DIR")
    if not env_drive_dir:
        print(
            "Drive project dir: disabled "
            "(set KEELNET_DRIVE_SYNC_DIR to a local Google Drive sync folder to mirror artifacts there)."
        )
        return None

    drive_project_dir = Path(env_drive_dir).expanduser().resolve()
    drive_project_dir.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir: {drive_project_dir}")
    return drive_project_dir


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    if not IS_HOSTED_COLAB:
        print("HF_TOKEN not set in the environment; continuing with anonymous HF access.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def resolve_local_repo_dir() -> Path:
    candidates: list[Path] = []
    env_repo_dir = os.environ.get("KEELNET_REPO_DIR")
    if env_repo_dir:
        candidates.append(Path(env_repo_dir).expanduser())
    candidates.append(DEFAULT_LOCAL_REPO_DIR)
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / ".git").exists() and (resolved / "pyproject.toml").exists():
            return resolved

    raise FileNotFoundError(
        "Could not find the KeelNet repo. Set KEELNET_REPO_DIR to your local checkout before running this cell."
    )


def ensure_repo() -> Path:
    if not IS_HOSTED_COLAB:
        return resolve_local_repo_dir()

    local_repo_dir = Path(os.environ.get("KEELNET_REPO_DIR", str(DEFAULT_LOCAL_REPO_DIR)))
    if (local_repo_dir / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=local_repo_dir)
    else:
        run_setup(["git", "clone", GIT_REPO_URL, str(local_repo_dir)])

    run_setup(["git", "checkout", GIT_BRANCH], cwd=local_repo_dir)
    run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=local_repo_dir)
    return local_repo_dir.resolve()


PROJECT_STORAGE_DIR = configure_project_storage()
DRIVE_PROJECT_DIR = configure_drive_project_dir(PROJECT_STORAGE_DIR)
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Runtime repo dir: {REPO_DIR}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


def mirror_output_root(output_root: Path) -> Path | None:
    if DRIVE_PROJECT_DIR is None:
        print("Drive artifact mirror is disabled for this runtime.")
        return None

    output_root = Path(output_root).expanduser().resolve()
    if not output_root.exists():
        print(f"Nothing to mirror yet: {output_root}")
        return None

    drive_project_dir = DRIVE_PROJECT_DIR.expanduser().resolve()
    try:
        relative_output = output_root.relative_to(PROJECT_STORAGE_DIR.expanduser().resolve())
    except ValueError:
        relative_output = Path("artifacts") / output_root.name
    drive_output_root = drive_project_dir / relative_output

    if output_root == drive_output_root:
        print(f"Artifacts already stored in Drive: {output_root}")
        return output_root

    drive_output_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output_root, drive_output_root, dirs_exist_ok=True)
    print(f"Mirrored artifacts to Drive: {drive_output_root}")
    return drive_output_root


<h2 style="color: #1d4ed8;">2. Configure The Run</h2>

Set `AUTHOR_NAME` to your name. This notebook builds the stage-specific `RUN_NAME` automatically as `yourname-stage6-v1`, `yourname-stage6-v2`, `yourname-stage6-v3`, and so on based on completed runs.

Review `TARGET_METRICS`, `SUGGESTED_MODULES`, `SMOKE_TEST_COMMANDS`, and `STAGE_COMMANDS` before you move into implementation.

This cell also prints the important values you should check before running stage commands.

It creates a small `RUN_STARTED.txt` file in the current run folder immediately, so you can confirm the output path is correct before training or evaluation finishes.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys

import torch

REPO_DIR = Path(REPO_DIR).resolve()
PROJECT_STORAGE_DIR = Path(PROJECT_STORAGE_DIR).resolve()
DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR).resolve() if DRIVE_PROJECT_DIR is not None else None

# Change only this for each teammate. The notebook builds the stage name and next version automatically.
AUTHOR_NAME = "yourname"
RUN_BASENAME = f"{AUTHOR_NAME}-stage6"
ARTIFACTS_ROOT = PROJECT_STORAGE_DIR / "artifacts" / "stage6_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"

STAGE_TITLE = 'Stage 6: Adaptive Constraint Balancing'
STAGE_OBJECTIVE = 'Only if needed, beat the strongest earlier fixed or constrained baseline by adapting the trade-off among answer quality, support, abstention, and confidence.'
TARGET_METRICS = ['trade-off curve quality', 'comparison against strongest earlier baselines', 'robustness across operating points']
IMPLEMENTATION_HINTS = ['input: the calibrated and controlled signals from the earlier stages', 'output: adaptive balancing or decision control', 'replace the strongest earlier baseline, not a straw-man baseline']
SUGGESTED_MODULES = ['keelnet.balance', 'keelnet.evaluate', 'keelnet.metrics']

# Fill these in when the stage code is ready.
RUN_SMOKE_TEST = False
SMOKE_TEST_COMMANDS = [
    # Example:
    # [sys.executable, "-m", "keelnet.some_module", "--help"],
]
STAGE_COMMANDS = [
    # Example:
    # [sys.executable, "-m", "keelnet.some_module", "--output-dir", str(OUTPUT_ROOT / "trial-1")],
]


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME

RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"stage={STAGE_TITLE}",
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"runtime_mode={RUNTIME_MODE}",
            f"repo_dir={REPO_DIR}",
            f"project_storage_dir={PROJECT_STORAGE_DIR}",
            f"git_branch={CURRENT_BRANCH}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Repo dir: {REPO_DIR}")
print(f"{PROJECT_STORAGE_LABEL}: {PROJECT_STORAGE_DIR}")
print(f"Drive project dir: {DRIVE_PROJECT_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Target metrics:", ", ".join(TARGET_METRICS))
print("Suggested modules:", ", ".join(SUGGESTED_MODULES))


def run(cmd):
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    with subprocess.Popen(
        rendered,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, rendered)


def run_many(commands, *, label: str) -> bool:
    if not commands:
        print(f"No commands configured for {label} yet.")
        return False

    for index, cmd in enumerate(commands, start=1):
        print(f"\n[{label} {index}/{len(commands)}]")
        run(cmd)
    return True


<h2 style="color: #1d4ed8;">3. Validate The Environment</h2>

Run the project tests before stage-specific work. This confirms the installed code path is at least minimally healthy inside the current runtime.


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])
                


<h2 style="color: #1d4ed8;">4. Optional Smoke Test</h2>

Use this cell only after you fill in `SMOKE_TEST_COMMANDS` in the config cell. Keep those commands tiny so you can catch path, dependency, and runtime issues before a full Stage 6 run.


In [ ]:
if RUN_SMOKE_TEST:
    ran_smoke = run_many(SMOKE_TEST_COMMANDS, label="smoke test")
    if not ran_smoke:
        print("Add one or more small commands to SMOKE_TEST_COMMANDS in the config cell.")
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.")
                


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Implementation Starts Here</strong><br/>
Sections 1-4 are setup and validation. Section 5 onward is the main Stage 6 work area.
<ul>
  <li><strong>Start here:</strong> create <code>src/keelnet/balance.py</code> or an equivalent adaptive-control module, then compare against the strongest earlier Stage 4 or Stage 5 baseline in evaluation.</li>
  <li><strong>Finish here:</strong> adaptive balancing beats the strongest earlier baseline on the main trade-offs.</li>
  <li><strong>Out of scope:</strong> restarting earlier stages, open-ended generation claims, general hallucination claims.</li>
</ul>
</div>


## IMPLEMENTATION 1: Run The Stage 6 Balancing Commands

This is the main Stage 6 implementation and run section. Everything before this point is setup, sync, or validation.

Fill in `STAGE_COMMANDS` in the config cell with the actual commands for this stage. Start with one command, make sure the outputs land in `OUTPUT_ROOT`, then add the rest.


In [ ]:
ran_stage = run_many(STAGE_COMMANDS, label="stage command")
if not ran_stage:
    print("Fill in STAGE_COMMANDS in the config cell before running this section.")


## Stage Note Template

Keep your stage notes inside this notebook flow. Update them at three points:

1. before implementation: fill in the goal, success condition, and planned commands
2. after smoke test: record environment issues and command fixes
3. after a meaningful run: record metrics, verdict, and next actions

Use this structure for the generated run note:

- Run info
- Goal
- Commands
- Main metrics
- What changed
- What worked
- What failed or looks risky
- Error cases to review
- Decision
- Next actions


<h2 style="color: #15803d;">Save Notes And Review Artifacts</h2>

This cell creates teammate-friendly note files inside the current run folder and lists the current artifacts. Update the generated notes as you learn what does and does not work in Stage 6: Adaptive Constraint Balancing.


In [ ]:
mirrored_output_root = mirror_output_root(OUTPUT_ROOT)

if not RUN_NOTES_FILE.exists():
    metric_lines = [f"- {metric}" for metric in TARGET_METRICS]
    RUN_NOTES_FILE.write_text(
        "\n".join(
            [
                f"# {STAGE_TITLE} Notes",
                "",
                "Update this note three times:",
                "1. before implementation: goal, success condition, and commands",
                "2. after smoke test: environment issues and command fixes",
                "3. after a meaningful run: metrics, verdict, and next actions",
                "",
                "## Run Info",
                f"- Branch: {CURRENT_BRANCH}",
                f"- `RUN_NAME`: {RUN_NAME}",
                f"- Output folder: {OUTPUT_ROOT}",
                f"- Runtime mode: {RUNTIME_MODE}",
                "",
                "## Goal",
                f"- One-sentence objective: {STAGE_OBJECTIVE}",
                "- Success condition:",
                "- Out of scope:",
                "",
                "## Commands",
                f"- Smoke test command(s): {SMOKE_TEST_COMMANDS}",
                f"- Main command(s): {STAGE_COMMANDS}",
                "- Input artifacts or checkpoints:",
                "- Output files to inspect:",
                "",
                "## Main Metrics",
                *metric_lines,
                "",
                "## What Changed",
                "- ",
                "",
                "## What Worked",
                "- ",
                "",
                "## What Failed Or Looks Risky",
                "- ",
                "",
                "## Error Cases To Review",
                "- ",
                "",
                "## Decision",
                "- Keep, revise, or stop:",
                "- Reason:",
                "",
                "## Next Actions",
                "1. ",
                "2. ",
                "3. ",
            ]
        )
        + "\n",
        encoding="utf-8",
    )

RUN_SUMMARY_FILE.write_text(
    json.dumps(
        {
            "stage": STAGE_TITLE,
            "run_name": RUN_NAME,
            "runtime_mode": RUNTIME_MODE,
            "git_branch": CURRENT_BRANCH,
            "project_storage_dir": str(PROJECT_STORAGE_DIR),
            "drive_project_dir": str(DRIVE_PROJECT_DIR) if DRIVE_PROJECT_DIR is not None else None,
            "output_root": str(OUTPUT_ROOT),
            "mirrored_output_root": str(mirrored_output_root) if mirrored_output_root is not None else None,
            "target_metrics": TARGET_METRICS,
            "suggested_modules": SUGGESTED_MODULES,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
if mirrored_output_root is not None:
    print(f"Drive mirror: {mirrored_output_root}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


<h2 style="color: #15803d;">Final Check</h2>

Stage 6 is only interesting if it really beats simpler controls.

Check all three:

- the adaptive method beats the strongest fixed or constrained baseline you already trust
- the gain is not just threshold gaming or over-abstention
- the operating-point story is simple enough to explain to teammates and in the report

After that, save the comparison artifacts that make the baseline-versus-adaptive trade-off easy to defend.


<h2 style="color: #15803d;">Share This Run</h2>

This cell prints a minimal share-ready summary for teammates, saves it into the current run folder, and marks the run as completed so the next run becomes the next version.

Update the metric lines after you review the outputs for this stage.


In [ ]:
from datetime import datetime, timezone

mirrored_output_root = mirror_output_root(OUTPUT_ROOT)
share_lines = [
    f"# {STAGE_TITLE} Share Note",
    "",
    f"- runtime mode: {RUNTIME_MODE}",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    *[f"- {metric}: <fill in after review>" for metric in TARGET_METRICS],
    f"- Output folder path: {OUTPUT_ROOT}",
]
if mirrored_output_root is not None:
    share_lines.append(f"- Drive mirror path: {mirrored_output_root}")
share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
print(f"Update the metric lines in: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")
if mirrored_output_root is not None:
    mirror_output_root(OUTPUT_ROOT)
